In [4]:
import os
import traceback
from conllu import parse_incr

class Token:
    def __init__(self, id_, form, lemma, upos, xpos, feats, head, deprel, deps, misc):
        self.id = id_
        self.form = form
        self.lemma = lemma
        self.upos = upos
        self.xpos = xpos
        self.feats = feats if feats is not None else {}  #feats is a dictionary
        self.head = head
        self.deprel = deprel
        self.deps = deps
        self.misc = misc

    def __str__(self):
        return (f"Token(id={self.id}, form={self.form}, lemma={self.lemma}, "
                f"upos={self.upos}, xpos={self.xpos}, feats={self.feats}, "
                f"head={self.head}, deprel={self.deprel}, deps={self.deps}, "
                f"misc={self.misc})")

class Sentence:
    def __init__(self, sentence_id, tokens, metadata=None):
        self.sentence_id = sentence_id  # Unique ID for each sentence
        self.tokens = tokens  # List of Token objects
        self.metadata = metadata if metadata else {}  # Metadata like 'newpart' or any other sentence-level info

    def __str__(self):
        token_strs = ', '.join([str(token) for token in self.tokens])
        return f"Sentence(id={self.sentence_id}, tokens=[{token_strs}], metadata={self.metadata})"

    def get_tokens(self):
        return self.tokens

    def get_metadata(self):
        return self.metadata

def convert_tuple_id_to_float(id_tuple):
    """
    Convert a tuple (integer_before_point, decimal_position, integer_after_point)
    to a float number.
    """
    if isinstance(id_tuple, tuple) and len(id_tuple) == 3:
        integer_part = id_tuple[0]
        decimal_part = id_tuple[2]
        id_num = f"{integer_part}.{decimal_part}"  # Create float number as string
        return float(id_num)
    return float(id_tuple)  # In case it's already a float or int

def process_conllu_file(file_path):
    sentences = []  # List to store all Sentence objects
    try:
        with open(file_path, 'r', encoding='utf-8') as data_file:
            for idx, sentence in enumerate(parse_incr(data_file), 1):
                tokens_list = []
                metadata = sentence.metadata  # Extract metadata like 'newpart' if it exists

                # Process each token in the sentence
                for token_data in sentence:
                    token_id = token_data['id']
                    if isinstance(token_id, tuple):
                        token_id = convert_tuple_id_to_float(token_id)

                    token = Token(
                        id_=token_id,
                        form=token_data['form'],
                        lemma=token_data['lemma'],
                        upos=token_data['upos'],
                        xpos=token_data['xpos'],
                        feats=token_data.get('feats', {}),
                        head=token_data['head'],
                        deprel=token_data['deprel'],
                        deps=token_data['deps'],
                        misc=token_data['misc']
                    )
                    tokens_list.append(token)

                # Create Sentence object with its tokens and metadata
                sentence_obj = Sentence(sentence_id=idx, tokens=tokens_list, metadata=metadata)
                sentences.append(sentence_obj)
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        traceback.print_exc()

    return sentences

def process_folder(folder_path):
    all_sentences = []  # List to store all sentences across multiple files
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".conllu"):
                file_path = os.path.join(root, file)
                sentences = process_conllu_file(file_path)
                all_sentences.extend(sentences)
    return all_sentences

def load_sentences_from_folder(folder_path):
    """
    This function processes all .conllu files in a folder and returns:
    - A list of all Sentence objects
    - A flat list of all Token objects
    """
    sentences = process_folder(folder_path)
    all_tokens = [token for sentence in sentences for token in sentence.get_tokens()]  # Flatten all tokens from all sentences
    return sentences, all_tokens

def total_sentences_and_tokens():
    folder_path = 'C:\\Users\\rahaa\\Dropbox\\MPCD\\codes\\output'
    sentences, tokens = load_sentences_from_folder(folder_path)

    print(f"Total sentences found: {len(sentences)}")
    print(f"Total tokens found: {len(tokens)}")

    # Example of how to use the corpus:
    # For instance, print the first sentence and its tokens
    if sentences:
        print(f"First sentence: {sentences[0]}")
        print(f"Tokens in the first sentence: {[str(token) for token in sentences[0].get_tokens()]}")
    return sentences
if __name__ == "__main__":
    total_sentences_and_tokens()


Total sentences found: 2671
Total tokens found: 48730
First sentence: Sentence(id=1, tokens=[Token(id=1, form=pursišnīhā, lemma=pursišn, upos=NOUN, xpos=None, feats={'Animacy': 'Inan', 'Number': 'Plur'}, head=0, deprel=root, deps=[('obj', 12), ('obj', 19)], misc={'question': ''}), Token(id=2, form=čand, lemma=čand, upos=DET, xpos=None, feats={'PronType': 'Ind'}, head=3, deprel=det, deps=None, misc={'some': ''}), Token(id=3, form=dar, lemma=dar, upos=NOUN, xpos=None, feats={'Animacy': 'Inan'}, head=1, deprel=nmod, deps=None, misc={'topic': ''}), Token(id=4, form=ī, lemma=ī, upos=SCONJ, xpos=None, feats={'PronType': 'Rel'}, head=12, deprel=mark, deps=[('ref', 1)], misc={'which': ''}), Token(id=5, form=mihrxwaršēd, lemma=mihrxwaršēd, upos=PROPN, xpos=None, feats={'Animacy': 'Hum', 'Gender': 'Masc', 'NameType': 'Giv'}, head=12, deprel=nsubj, deps=None, misc={'Mihrxwaršēd': ''}), Token(id=6, form=ī, lemma=ī, upos=DET, xpos=None, feats={}, head=7, deprel=det, deps=None, misc={'ezafe': ''}), 

In [5]:
def analyze_all_sentences(sentences):
    """
    This function analyzes all sentences, regardless of the type of head, and looks for subjects.
    """
    # Sort the sentences alphabetically by sentence_id
    sentences.sort(key=lambda clause: clause.metadata.get('sent_id', ''))

    # Analyze all sentences
    print("Analysis of all sentences:")
    for clause in sentences:
        analyze_clause(clause)

def analyze_clause(clause):
    # Set a flag to only print the first subject per sentence (if needed)
    found_subject = False

    # Iterate over each token in the sentence using get_tokens()
    for token in clause.get_tokens():
        # Check if the token is a subject but not a NOUN, PRON, PROPN, or NUM
        if token.deprel == 'nsubj' and token.upos not in ['NOUN', 'PRON', 'PROPN', 'NUM']:
            if not found_subject:
                # Print information about the first found subject
                print(f"Subject UPOS: {token.upos}, Subject Form: {token.form} | Newpart: {clause.metadata.get('newpart', 'N/A')} | "
                      f"Sentence id: {clause.metadata.get('sent_id', 'N/A')} | Text: {clause.metadata.get('text', 'N/A')}")
                found_subject = True  # Mark that we've found a subject and printed it

if __name__ == "__main__":
    sentences, _ = load_sentences_from_folder (folder_path='C:\\Users\\rahaa\\Dropbox\\MPCD\\conllus_with_erros')
    analyze_all_sentences(sentences)


NameError: name 'sentences' is not defined